# 04 - Training & evaluation across all split schemes - CSE-CIC-IDS2018

In [1]:
DATASET = "cic2018"
FT_ROOT = None
TARGET_PER_CLASS = 50_000
MODELS_TO_RUN = None
RUN_CV = True

In [2]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS
from ids.models.pipelines import build_pipeline
from ids.models.registry import MODEL_NAMES
from ids.evaluation.metrics import evaluate_model
from ids.evaluation.cross_val import cross_validate_models, summarize_cv
from ids.evaluation.plots import plot_confusion_matrix

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "04_experiments.log", "nb04")

df = pd.read_parquet(P.data / "clean.parquet")
X, y = df[CANONICAL_COLUMNS], df["y"].to_numpy()
top40 = nbc.load_json(P.features / "top40.json")["selected"]
mapping = nbc.load_json(P.data / "label_mapping.json")
class_names = [c for c, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
n_classes = len(class_names)
models_to_run = MODELS_TO_RUN or list(MODEL_NAMES)
log.info("%d rows | %d selected features | %d classes | models=%s | target_per_class=%d",
         len(X), len(top40), n_classes, models_to_run, TARGET_PER_CLASS)

2026-07-06 09:21:10,608 INFO nb04: 1586902 rows | 40 selected features | 15 classes | models=['decision_tree', 'random_forest', 'svm', 'naive_bayes', 'knn'] | target_per_class=50000


## 1. Build all split-scheme indices

In [3]:
idx = np.arange(len(X))

def holdout(test_size):
    tr, te = train_test_split(idx, test_size=test_size, stratify=y,
                              random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "test_idx": te}

def train_val_test():
    tr_val, te = train_test_split(idx, test_size=0.10, stratify=y,
                                  random_state=config.RANDOM_STATE)
    tr, val = train_test_split(tr_val, test_size=2/9, stratify=y[tr_val],
                               random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "val_idx": val, "test_idx": te}

SPLITS = {}
p_7030 = P.splits / "70_30_indices.npz"
SPLITS["70_30"] = dict(np.load(p_7030)) if p_7030.exists() else holdout(0.30)
SPLITS["80_20"] = holdout(0.20)
SPLITS["90_10"] = holdout(0.10)
SPLITS["70_20_10"] = train_val_test()

for scheme, s in SPLITS.items():
    np.savez_compressed(P.splits / f"{scheme}_indices.npz", **s)
    log.info("%s: %s", scheme,
             " / ".join(f"{k.replace('_idx','')}={len(v)}" for k, v in s.items()))

2026-07-06 09:21:12,676 INFO nb04: 70_30: train=1110831 / test=476071
2026-07-06 09:21:13,488 INFO nb04: 80_20: train=1269521 / test=317381
2026-07-06 09:21:14,273 INFO nb04: 90_10: train=1428211 / test=158691
2026-07-06 09:21:15,059 INFO nb04: 70_20_10: train=1110830 / val=317381 / test=158691


## 2. Train and evaluate every scheme x model combination

In [4]:
def run_one(scheme, name, train_idx, test_idx, val_idx=None):
    out_dir = P.runs / scheme / name
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.json"
    if metrics_path.exists():
        log.info("SKIP %s/%s — metrics.json exists", scheme, name)
        return nbc.load_json(metrics_path)

    pipeline = build_pipeline(name, top40, target_per_class=TARGET_PER_CLASS)
    t0 = time.perf_counter()
    pipeline.fit(X.iloc[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - t0
    log.info("%s/%s: fit on %d rows in %.1fs", scheme, name, len(train_idx), fit_seconds)

    record = {"dataset": DATASET, "scheme": scheme, "model": name,
              "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
              "target_per_class": TARGET_PER_CLASS, "fit_seconds": round(fit_seconds, 3)}
    evals = [("test", test_idx, "")]
    if val_idx is not None:
        record["n_val"] = int(len(val_idx))
        evals.append(("validation", val_idx, "_val"))

    for split_name, eidx, suffix in evals:
        m, _, y_pred, y_proba = evaluate_model(name, pipeline, X.iloc[eidx], y[eidx])
        cm = confusion_matrix(y[eidx], y_pred, labels=np.arange(n_classes))
        np.save(out_dir / f"confusion_matrix{suffix}.npy", cm)
        np.save(out_dir / f"predictions{suffix}.npy", y_pred)
        if y_proba is not None:
            np.save(out_dir / f"probas{suffix}.npy", y_proba.astype(np.float32))
        rep = classification_report(y[eidx], y_pred, labels=np.arange(n_classes),
                                    target_names=class_names, output_dict=True,
                                    zero_division=0)
        pd.DataFrame(rep).T.to_csv(out_dir / f"per_class{suffix}.csv")
        if split_name == "test":
            plot_confusion_matrix(f"{name} ({scheme})", cm, class_names, out_dir)
        record[split_name] = m.to_dict()
        log.info("%s/%s [%s]: acc=%.4f f1_macro=%.4f roc_auc=%s det=%.3fms",
                 scheme, name, split_name, m.accuracy, m.f1_macro,
                 "n/a" if m.roc_auc is None else round(m.roc_auc, 4),
                 m.detection_time_ms)

    nbc.save_json(record, metrics_path)
    return record

records = []
with nbc.timed(log, "all holdout schemes"):
    for scheme, s in SPLITS.items():
        with nbc.timed(log, f"scheme {scheme}"):
            for name in models_to_run:
                records.append(run_one(scheme, name, s["train_idx"], s["test_idx"],
                                       s.get("val_idx")))
pd.DataFrame([{ "scheme": r["scheme"], "model": r["model"],
                "fit_s": r["fit_seconds"], **{k: round(v, 4) if isinstance(v, float) else v
                for k, v in r["test"].items() if k not in ("name",)}} for r in records])

2026-07-06 09:21:15,073 INFO nb04: START all holdout schemes
2026-07-06 09:21:15,074 INFO nb04: START scheme 70_30
2026-07-06 09:21:34,233 INFO nb04: 70_30/decision_tree: fit on 1110831 rows in 19.2s
2026-07-06 09:21:36,728 INFO nb04: 70_30/decision_tree [test]: acc=0.7258 f1_macro=0.6684 roc_auc=0.9231 det=0.000ms


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.6s


2026-07-06 09:22:16,727 INFO nb04: 70_30/random_forest: fit on 1110831 rows in 40.0s


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   34.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    3.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    2.2s finished


2026-07-06 09:22:27,242 INFO nb04: 70_30/random_forest [test]: acc=0.7537 f1_macro=0.6912 roc_auc=0.9503 det=0.007ms
2026-07-06 09:22:37,435 INFO nb04: 70_30/svm: fit on 1110831 rows in 10.2s
2026-07-06 09:22:41,203 INFO nb04: 70_30/svm [test]: acc=0.4789 f1_macro=0.4901 roc_auc=0.8707 det=0.001ms
2026-07-06 09:22:46,186 INFO nb04: 70_30/naive_bayes: fit on 1110831 rows in 5.0s
2026-07-06 09:22:51,107 INFO nb04: 70_30/naive_bayes [test]: acc=0.4082 f1_macro=0.4384 roc_auc=0.8623 det=0.001ms
2026-07-06 09:22:56,289 INFO nb04: 70_30/knn: fit on 1110831 rows in 5.2s
2026-07-06 09:33:59,442 INFO nb04: 70_30/knn [test]: acc=0.7209 f1_macro=0.6471 roc_auc=0.9272 det=0.474ms
2026-07-06 09:33:59,458 INFO nb04: END   scheme 70_30 (764.4s)
2026-07-06 09:33:59,458 INFO nb04: START scheme 80_20
2026-07-06 09:34:18,341 INFO nb04: 80_20/decision_tree: fit on 1269521 rows in 18.9s
2026-07-06 09:34:20,183 INFO nb04: 80_20/decision_tree [test]: acc=0.6937 f1_macro=0.6536 roc_auc=0.9164 det=0.000ms


...

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   31.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.6s finished


2026-07-06 09:48:44,280 INFO nb04: 70_20_10/random_forest [test]: acc=0.7535 f1_macro=0.6625 roc_auc=0.95 det=0.005ms


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.3s finished


2026-07-06 09:48:49,246 INFO nb04: 70_20_10/random_forest [validation]: acc=0.7524 f1_macro=0.6944 roc_auc=0.9498 det=0.005ms
2026-07-06 09:49:00,347 INFO nb04: 70_20_10/svm: fit on 1110830 rows in 11.1s
2026-07-06 09:49:01,580 INFO nb04: 70_20_10/svm [test]: acc=0.4714 f1_macro=0.4927 roc_auc=0.8747 det=0.001ms
2026-07-06 09:49:03,766 INFO nb04: 70_20_10/svm [validation]: acc=0.4711 f1_macro=0.4962 roc_auc=0.8746 det=0.000ms
2026-07-06 09:49:08,211 INFO nb04: 70_20_10/naive_bayes: fit on 1110830 rows in 4.4s
2026-07-06 09:49:09,442 INFO nb04: 70_20_10/naive_bayes [test]: acc=0.4920 f1_macro=0.4467 roc_auc=0.8603 det=0.001ms
2026-07-06 09:49:11,681 INFO nb04: 70_20_10/naive_bayes [validation]: acc=0.4934 f1_macro=0.4486 roc_auc=0.8609 det=0.001ms
2026-07-06 09:49:16,051 INFO nb04: 70_20_10/knn: fit on 1110830 rows in 4.4s
2026-07-06 09:52:56,516 INFO nb04: 70_20_10/knn [test]: acc=0.7224 f1_macro=0.6449 roc_auc=0.9285 det=0.458ms
2026-07-06 09:59:57,732 INFO nb04: 70_20_10/knn [validat

,scheme,model,fit_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,detection_time_ms,n_test
0,70_30,decision_tree,19.157,0.7258,0.6510,0.7838,0.6684,0.9231,0.0003,476071
1,70_30,random_forest,39.997,0.7537,0.6826,0.7889,0.6912,0.9503,0.0070,476071
2,70_30,svm,10.164,0.4789,0.4572,0.7926,0.4901,0.8707,0.0010,476071
3,70_30,naive_bayes,4.983,0.4082,0.4360,0.7004,0.4384,0.8623,0.0011,476071
4,70_30,knn,5.179,0.7209,0.6313,0.7912,0.6471,0.9272,0.4743,476071
5,80_20,decision_tree,18.881,0.6937,0.6400,0.7813,0.6536,0.9164,0.0003,317381
6,80_20,random_forest,34.995,0.7523,0.6669,0.8058,0.6838,0.9507,0.0053,317381
7,80_20,svm,10.201,0.4452,0.4925,0.8036,0.5150,0.8782,0.0011,317381
8,80_20,naive_bayes,3.871,0.4280,0.4486,0.7150,0.4514,0.8615,0.0007,317381
9,80_20,knn,3.570,0.7210,0.6373,0.8059,0.6547,0.9276,0.4678,317381


## 3. Five-fold stratified cross-validation (scheme `cv5`)

In [5]:
cv_dir = P.runs / "cv5"
cv_dir.mkdir(parents=True, exist_ok=True)
cv_scores_path = cv_dir / "cv_scores.csv"

if not RUN_CV:
    log.info("RUN_CV=False — skipping")
elif cv_scores_path.exists():
    log.info("SKIP cv5 — cv_scores.csv exists")
    cv_summary = pd.read_csv(cv_dir / "cv_summary.csv")
else:
    with nbc.timed(log, "5-fold stratified CV, all models"):
        scores, _ = cross_validate_models(X, y, top40, model_names=models_to_run,
                                          n_splits=5, target_per_class=TARGET_PER_CLASS)
    scores.to_csv(cv_scores_path, index=False)
    cv_summary = summarize_cv(scores)
    cv_summary.to_csv(cv_dir / "cv_summary.csv", index=False)
cv_summary if RUN_CV else None

2026-07-06 09:59:57,793 INFO nb04: START 5-fold stratified CV, all models
2026-07-06 10:00:15,849 INFO ids.evaluation.cross_val: CV decision_tree fold 1/5: f1_macro=0.6788 acc=0.7116 (16.6s)
2026-07-06 10:00:32,752 INFO ids.evaluation.cross_val: CV decision_tree fold 2/5: f1_macro=0.6759 acc=0.7276 (15.7s)
2026-07-06 10:00:50,093 INFO ids.evaluation.cross_val: CV decision_tree fold 3/5: f1_macro=0.6573 acc=0.7029 (16.1s)
2026-07-06 10:01:06,948 INFO ids.evaluation.cross_val: CV decision_tree fold 4/5: f1_macro=0.6618 acc=0.7302 (15.9s)
2026-07-06 10:01:24,415 INFO ids.evaluation.cross_val: CV decision_tree fold 5/5: f1_macro=0.6649 acc=0.7225 (16.4s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   11.9s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   30.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 10:02:02,250 INFO ids.evaluation.cross_val: CV random_forest fold 1/5: f1_macro=0.6571 acc=0.7427 (34.2s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   11.4s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   29.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 10:02:38,973 INFO ids.evaluation.cross_val: CV random_forest fold 2/5: f1_macro=0.6186 acc=0.7409 (33.1s)


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   31.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 10:03:59,941 INFO ids.evaluation.cross_val: CV random_forest fold 4/5: f1_macro=0.6781 acc=0.7407 (35.1s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   11.4s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   29.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.6s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


2026-07-06 10:04:36,411 INFO ids.evaluation.cross_val: CV random_forest fold 5/5: f1_macro=0.6860 acc=0.7390 (32.5s)
2026-07-06 10:04:49,644 INFO ids.evaluation.cross_val: CV svm fold 1/5: f1_macro=0.4624 acc=0.4779 (11.2s)
2026-07-06 10:05:00,189 INFO ids.evaluation.cross_val: CV svm fold 2/5: f1_macro=0.4716 acc=0.4784 (8.5s)
2026-07-06 10:05:10,647 INFO ids.evaluation.cross_val: CV svm fold 3/5: f1_macro=0.4574 acc=0.4731 (8.8s)
2026-07-06 10:05:21,216 INFO ids.evaluation.cross_val: CV svm fold 4/5: f1_macro=0.4704 acc=0.4717 (8.5s)
2026-07-06 10:05:31,483 INFO ids.evaluation.cross_val: CV svm fold 5/5: f1_macro=0.4734 acc=0.4874 (8.6s)
2026-07-06 10:05:36,725 INFO ids.evaluation.cross_val: CV naive_bayes fold 1/5: f1_macro=0.4145 acc=0.2588 (3.5s)
2026-07-06 10:05:41,761 INFO ids.evaluation.cross_val: CV naive_bayes fold 2/5: f1_macro=0.4473 acc=0.4894 (3.3s)
2026-07-06 10:05:46,787 INFO ids.evaluation.cross_val: CV naive_bayes fold 3/5: f1_macro=0.3926 acc=0.1734 (3.4s)
2026-07-06

,model,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,roc_auc_mean,roc_auc_std
0,decision_tree,0.718948,0.011460,0.654999,0.016089,0.796711,0.028801,0.667736,0.009233,0.916579,0.003056
1,knn,0.718657,0.001703,0.623007,0.014353,0.816061,0.028705,0.640211,0.015018,0.926850,0.000676
2,naive_bayes,0.305903,0.115585,0.418197,0.017758,0.668545,0.032820,0.414579,0.020283,0.865645,0.002112
3,random_forest,0.741071,0.001407,0.645248,0.022692,0.801094,0.034093,0.658038,0.026449,0.950343,0.000286
4,svm,0.477698,0.006144,0.436205,0.004246,0.769935,0.024248,0.467055,0.006833,0.868268,0.015836


In [6]:
n_files = sum(1 for _ in P.runs.rglob("*") if _.is_file())
log.info("notebook 04 complete — %d artifact files under %s", n_files, P.runs)

2026-07-06 10:30:10,290 INFO nb04: notebook 04 complete — 142 artifact files under /Users/MAC/Projects/IDS/backend/artifacts/ft40/cic2018/runs
